# Inequality Analysis — Joint Digital & the Middle-Income Trap

Measures inequality **three ways** the data can actually carry it:
1. **Structural inequality** (the dual economy) — ranks countries, draws the map
2. **Displacement inequality** — built from the *time-varying* AI series so it moves (unlike the Gini)
3. **Does AI create inequality?** — AI→Gini test, plus a clean **labor-share** test (auto-runs if you add PWT `labsh`)

**How to use:** `Runtime → Run all`. In Step 0 upload your panel CSV (`final_panel_imputed__5_.csv`).
*Optional:* also upload `pwt1001.xlsx` (Penn World Table) to unlock the clean AI→inequality test.

### Setup — install dependencies

In [ ]:
!pip install linearmodels country_converter -q
import pandas as pd, numpy as np, warnings, io
warnings.filterwarnings("ignore")
from linearmodels.panel import PanelOLS, BetweenOLS
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
print("ready")

### Step 0 — Upload your files
Click **Choose Files** and upload your panel (`final_panel_imputed__5_.csv`).
Optionally also upload `pwt1001.xlsx` to activate the labor-share test.

In [ ]:
from google.colab import files
up = files.upload()

# auto-detect the panel (the CSV with iso3 + year)
PANEL_FILE = None
for fn in up:
    if fn.lower().endswith(".csv"):
        try:
            head = pd.read_csv(fn, nrows=5)
            if {"iso3","year"}.issubset(head.columns):
                PANEL_FILE = fn; break
        except Exception:
            pass
if PANEL_FILE is None:
    raise SystemExit("No panel CSV with iso3+year columns found. Re-run and upload final_panel_imputed__5_.csv")
print("panel detected:", PANEL_FILE)

df = pd.read_csv(PANEL_FILE, low_memory=False).sort_values(["iso3","year"])

# optional: merge PWT labor share if an xlsx was uploaded
PWT_FILE = next((fn for fn in up if fn.lower().endswith(".xlsx")), None)
if PWT_FILE:
    try:
        pwt = pd.read_excel(PWT_FILE, sheet_name="Data")[["countrycode","year","labsh"]]
        pwt = pwt.rename(columns={"countrycode":"iso3"})
        df = df.merge(pwt, on=["iso3","year"], how="left")
        print(f"merged PWT labor share: {df['labsh'].notna().sum()} non-null rows -> labor-share test ENABLED")
    except Exception as e:
        print("PWT merge failed:", e)
else:
    print("no PWT file uploaded -> labor-share test will stay idle (that's fine)")

print("shape:", df.shape, "| countries:", df.iso3.nunique(), "| years:", df.year.min(), "-", df.year.max())

### Step 1 — Build the inequality measures

In [ ]:
df["G_fe"] = df.groupby("year")["ai_pubs_log"].transform(lambda s:(s-s.mean())/s.std())
g01        = df.groupby("year")["ai_pubs_log"].transform(lambda s:(s-s.min())/(s.max()-s.min()))
df["DualGap"] = df["sigma_kis"]*(1-df["H"])                 # STRUCTURAL inequality
df["Dexp"]    = g01*(1-df["H"])*(1-df["sigma_kis"])         # DISPLACEMENT exposure (moves)
for c in ["G_fe","Dexp","F"]:
    df[c+"_l"] = df.groupby("iso3")[c].shift(1)
df["FG_l"] = df["F_l"]*df["G_fe_l"]
df["GH_l"] = df["G_fe_l"]*(1-df["H"])                       # skill-biased AI term
CONTROLS=[c for c in ["log_gdp_l","trade_pct_gdp_l","investment_pct_gdp_l",
          "pop_growth_l","credit_to_private_pct_gdp_l"] if c in df.columns]

def stars(p): return "***" if p<.01 else "**" if p<.05 else "*" if p<.1 else ""
def fe(dep,regs):
    d=df.set_index(["iso3","year"])[[dep]+regs].dropna()
    f=f"{dep} ~ "+" + ".join(regs)+" + EntityEffects + TimeEffects"
    r=PanelOLS.from_formula(f,data=d,drop_absorbed=True).fit(cov_type="clustered",cluster_entity=True)
    return r,d.index.get_level_values(0).nunique()
def btw(dep,regs):
    d=df.set_index(["iso3","year"])[[dep]+regs].dropna()
    return BetweenOLS.from_formula(f"{dep} ~ 1 + "+" + ".join(regs),data=d).fit()
def wb(col):
    w=df.groupby("iso3")[col].std().median(); b=df.groupby("iso3")[col].mean().std()
    return (w/b if b else np.nan)
print("measures built. DualGap, Dexp, lags ready.")

### Step 2 — Part A: Structural inequality (ranked middle-income countries)

In [ ]:
recent=df[df.year>=2019]
agg=recent.groupby("iso3").agg(income_group=("income_group","last"),H=("H","mean"),
     services=("sigma_kis","mean"),DualGap=("DualGap","mean")).reset_index()
mi=agg[agg.income_group.isin(["UM","LM"])].dropna(subset=["DualGap"]).sort_values("DualGap",ascending=False)
mi.insert(0,"rank",range(1,len(mi)+1))
print(f"Middle-income countries ranked by STRUCTURAL inequality (top 15 of {len(mi)}):\n")
print(mi[["rank","iso3","H","services","DualGap"]].head(15).to_string(index=False))
agg.to_csv("country_structural_inequality.csv",index=False)
mi.to_csv("middle_income_structural_inequality.csv",index=False)

### Step 3 — Part B: Displacement inequality (does it move? does it drag growth?)

In [ ]:
print("Does each inequality metric MOVE within country? (need > 0.10 for fixed effects)\n")
for col in ["gini_disp"]+(["D_term"] if "D_term" in df.columns else [])+["Dexp"]:
    rt=wb(col); print(f"   {col:<10} within/between = {rt:.3f}   {'MOVES' if rt>0.10 else 'frozen'}")

cors=[]
for iso,g in df.groupby("iso3"):
    g=g.dropna(subset=["Dexp","G_fe"])
    if len(g)>=6 and g.Dexp.std()>0 and g.G_fe.std()>0: cors.append(g.Dexp.corr(g.G_fe))
print(f"\n   corr(displacement, AI) within country = {np.nanmedian(cors):+.2f}  (exposure index, not AI-independent)")

print("\nDisplacement drag on growth (theory wants Dexp_l < 0):")
r,nc=fe("gdp_pc_growth",["F_l","G_fe_l","FG_l","Dexp_l"]+CONTROLS)
for k in ["FG_l","Dexp_l"]:
    if k in r.params.index:
        print(f"   {k:<8}{r.params[k]:+8.4f}  se={r.std_errors[k]:.3f}  p={r.pvalues[k]:.3f} {stars(r.pvalues[k])}")
if "Slowdown" in df.columns:
    r2,_=fe("Slowdown",["F_l","G_fe_l","Dexp_l"]+CONTROLS)
    if "Dexp_l" in r2.params.index:
        print(f"   Dexp->slowdown {r2.params['Dexp_l']:+.4f}  p={r2.pvalues['Dexp_l']:.3f} {stars(r2.pvalues['Dexp_l'])}")
print("\n   Signs are theory-consistent but n.s. — displacement is ~0.9 collinear with AI.")

### Step 4 — Part C: Does AI *create* inequality?

In [ ]:
print("AI -> income inequality (Gini):\n")
for dep in ["gini_disp","gini"]:
    if dep not in df.columns: continue
    r,nc=fe(dep,["G_fe_l","GH_l","F_l"]+CONTROLS)
    rb=btw(dep,["G_fe","DualGap","F"])
    print(f"  {dep}:")
    print(f"     WITHIN  AI={r.params.get('G_fe_l',np.nan):+.4f} p={r.pvalues.get('G_fe_l',1):.3f} {stars(r.pvalues.get('G_fe_l',1))}"
          f" | skill-bias={r.params.get('GH_l',np.nan):+.4f} p={r.pvalues.get('GH_l',1):.3f}")
    print(f"     BETWEEN AI={rb.params.get('G_fe',np.nan):+.4f} p={rb.pvalues.get('G_fe',1):.3f} {stars(rb.pvalues.get('G_fe',1))} (neg = development confound)")
print(f"\n  Gini within/between = {wb('gini_disp'):.3f} -> within test underpowered by construction.")
print("  VERDICT: data do NOT show AI creating income inequality (within null; between wrong-signed).")

print("\n" + "-"*60)
print("CLEAN test on LABOR SHARE (non-circular):")
if "labsh" in df.columns and df["labsh"].notna().sum()>200:
    r,nc=fe("labsh",["G_fe_l","GH_l","F_l"]+CONTROLS)
    print(f"   AI -> labor share (within FE, N={r.nobs}, {nc} ctry):")
    for k in ["G_fe_l","GH_l","F_l"]:
        if k in r.params.index: print(f"      {k:<8}{r.params[k]:+8.4f}  p={r.pvalues[k]:.3f} {stars(r.pvalues[k])}")
    print("   theory: AI<0 (lowers labor's share = raises inequality), skill-bias<0.")
else:
    print("   [idle] upload pwt1001.xlsx in Step 0 to activate this clean AI->inequality test.")

### Step 5 — The dual-economy map

In [ ]:
Hmed=recent["H"].median(); Smed=recent["sigma_kis"].median()
# panel's own LH membership (drives the FG=+5.8 result)
def members(tag):
    col=f"FG_l_{tag}"
    return set(df.loc[df[col].fillna(0)!=0,"iso3"].unique()) if col in df.columns else set()
quad_of={}
for tag in ["HH","HL","LH","LL"]:
    for iso in members(tag): quad_of[iso]=tag
mi2=mi.copy(); mi2["LH"]=mi2["iso3"].map(quad_of).eq("LH")
try:
    import country_converter as cc
    mi2["country"]=cc.convert(mi2["iso3"].tolist(),to="name_short")
except Exception:
    mi2["country"]=mi2["iso3"]

fig,ax=plt.subplots(figsize=(11,8))
ax.add_patch(Rectangle((0,Smed),Hmed,1,facecolor="#2e7d4f",alpha=0.08,zorder=0))
ax.axvline(Hmed,color="#888",lw=1,ls="--"); ax.axhline(Smed,color="#888",lw=1,ls="--")
lh=mi2["LH"].fillna(False)
ax.scatter(mi2.loc[~lh,"H"],mi2.loc[~lh,"services"],s=60,c="#9aa0a6",alpha=.7,edgecolor="white",zorder=3,label="other middle-income")
ax.scatter(mi2.loc[lh,"H"],mi2.loc[lh,"services"],s=160,c="#2e7d4f",alpha=.9,edgecolor="white",zorder=4,label="dual economy (joint digital pays off)")
for _,r in mi2.iterrows():
    ax.annotate(r["country"],(r["H"],r["services"]),fontsize=7.5,xytext=(4,3),textcoords="offset points",
                color="#1a4d2e" if r["LH"] else "#444")
ax.text(Hmed*0.42,Smed*2.05,"DUAL ECONOMY\nlow skills · big modern\nservices sector",ha="center",fontsize=9,color="#2e7d4f",fontweight="bold")
ax.set_xlabel("Human capital (H)  →  more skilled workforce")
ax.set_ylabel("Knowledge-intensive services share (σ_KIS)  →  bigger modern sector")
ax.set_title("Structural inequality of middle-income countries\nDual economy (top-left): a modern sector its workforce can't fully access")
ax.legend(loc="lower right",fontsize=9); ax.set_ylim(0.01,max(0.10,mi2["services"].max()*1.05))
plt.tight_layout(); plt.savefig("fig_dual_economy_map.png",dpi=140); plt.show()

### Step 6 — Download the outputs

In [ ]:
from google.colab import files
for f in ["middle_income_structural_inequality.csv","country_structural_inequality.csv","fig_dual_economy_map.png"]:
    try: files.download(f)
    except Exception as e: print("skip",f,e)
print("done — check your downloads folder")